# ITECH IT6412 — Channel 1 Control
Use these cells to discover the instrument, connect via VISA (pyvisa), and set CH1 voltage/output. A simulator is provided if no device is found.


In [1]:
import time
from typing import Optional, Tuple

try:
    import pyvisa
except Exception:
    pyvisa = None

IDN_KEYWORD = 'ITECH'  # heuristic for auto-detect via *IDN?

class SimIT6412:
    def __init__(self):
        self.v1 = 0.0
        self.i1 = 0.5
        self.out1 = False
    def write(self, cmd: str):
        c = cmd.strip().upper()
        if c.startswith('APPL CH1'):
            # APPL CH1, V[, I]
            try:
                rest = cmd.split(' ', 2)[-1]
                parts = rest.split(',')
                if len(parts) >= 2:
                    self.v1 = float(parts[1])
                if len(parts) >= 3:
                    self.i1 = float(parts[2])
            except Exception:
                pass
        elif c.startswith('VOLT'):
            try:
                self.v1 = float(cmd.split()[-1])
            except Exception:
                pass
        elif c.startswith('SOUR:VOLT'):
            try:
                self.v1 = float(cmd.split()[-1])
            except Exception:
                pass
        elif c.startswith('OUTP CH1,'):
            self.out1 = ('ON' in c)
        elif c == 'OUTP ON' or c == 'OUTP 1':
            self.out1 = True
        elif c == 'OUTP OFF' or c == 'OUTP 0':
            self.out1 = False
    def query(self, cmd: str) -> str:
        c = cmd.strip().upper()
        if c == '*IDN?':
            return 'ITECH,IT6412,SIM,1.0'
        if c == 'VOLT?':
            return f'{self.v1}'
        if c == 'OUTP?':
            return '1' if self.out1 else '0'
        return ''
    def close(self):
        pass

class VisaIT6412:
    def __init__(self, resource: str):
        if pyvisa is None:
            raise RuntimeError('pyvisa not available')
        self.rm = pyvisa.ResourceManager()
        self.inst = self.rm.open_resource(resource)
        try:
            self.inst.timeout = 3000
        except Exception:
            pass
    def idn(self) -> str:
        try:
            return self.inst.query('*IDN?').strip()
        except Exception:
            return ''
    def _try(self, cmd: str) -> bool:
        try:
            self.inst.write(cmd)
            return True
        except Exception:
            return False
    def select_ch1(self):
        self._try('INST CH1')
    def set_ch1_voltage(self, volts: float, current_limit: Optional[float] = None) -> None:
        self.select_ch1()
        # Try common SCPI variants; stop on first success
        if current_limit is not None:
            if self._try(f'APPL CH1,{volts},{current_limit}'):
                return
        if self._try(f'APPL CH1,{volts}'):
            return
        if self._try(f'SOUR:VOLT {volts}'):
            return
        if self._try(f'VOLT {volts}'):
            return
        raise RuntimeError('Failed to set voltage on CH1 via known commands')
    def set_ch1_output(self, on: bool) -> None:
        # Try channel-specific, then selected-channel forms
        if self._try(f'OUTP CH1,{'ON' if on else 'OFF'}'):
            return
        self.select_ch1()
        if self._try(f'OUTP {'ON' if on else 'OFF'}'):
            return
        if self._try(f'OUTP {1 if on else 0}'):
            return
        raise RuntimeError('Failed to set output on CH1')
    def close(self):
        try:
            self.inst.close()
        except Exception:
            pass
        try:
            self.rm.close()
        except Exception:
            pass

def list_resources() -> list:
    if pyvisa is None:
        return []
    rm = pyvisa.ResourceManager()
    try:
        return list(rm.list_resources())
    finally:
        try:
            rm.close()
        except Exception:
            pass

def find_it6412(resource_hint: Optional[str] = None) -> Tuple[Optional[str], list]:
    resources = list_resources()
    if resource_hint:
        return resource_hint, resources
    # Probe *IDN? to find an ITECH
    for r in resources:
        try:
            rm = pyvisa.ResourceManager() if pyvisa else None
            inst = rm.open_resource(r) if rm else None
            idn = inst.query('*IDN?').strip() if inst else ''
            if inst: inst.close()
            if rm: rm.close()
            if IDN_KEYWORD in idn.upper():
                return r, resources
        except Exception:
            try:
                if inst: inst.close()
            except Exception:
                pass
            try:
                if rm: rm.close()
            except Exception:
                pass
    # Fallback: return first likely instrument path
    for r in resources:
        R = r.upper()
        if any(k in R for k in ['USB', 'GPIB', 'ASRL', 'TCPIP']):
            return r, resources
    return None, resources

def connect_it6412(resource_hint: Optional[str] = None, simulate_if_missing: bool = True):
    resource, resources = find_it6412(resource_hint)
    print('VISA resources:', resources)
    if resource and pyvisa is not None:
        print(f'Connecting to: {resource}')
        dev = VisaIT6412(resource)
        idn = dev.idn()
        print('IDN:', idn or '<unknown>')
        return dev
    if simulate_if_missing:
        print('No IT6412 found; using simulator.')
        return SimIT6412()
    raise RuntimeError('No IT6412 found and simulation disabled')


In [9]:
# Discover and connect (edit resource_hint if you know it)
psu = connect_it6412(resource_hint=None)
psu


VISA resources: ['USB0::0x2EC7::0x6412::800624011767410027::INSTR']
Connecting to: USB0::0x2EC7::0x6412::800624011767410027::INSTR
IDN: ITECH LTD, IT6412, 800624011767410027, ARM Rev 6412-1.20,Rev 6412-1.11,6412-1.11,LCD Rev 1.02


In [10]:
# Turn CH1 output ON
try:
    psu.set_ch1_output(True)
    print('CH1 output: ON')
except AttributeError:
    # Simulator path
    psu.write('OUTP CH1,ON')
    print('CH1 output: ON (sim)')


CH1 output: ON


In [ ]:
# Turn CH1 output OFF
try:
    psu.set_ch1_output(False)
    print('CH1 output: OFF')
except AttributeError:
    psu.write('OUTP CH1,OFF')
    print('CH1 output: OFF (sim)')


In [4]:
# Helper to set CH1 voltage
def set_ch1_voltage(volts: float, current_limit: float = None):
    print(f'Setting CH1 voltage to {volts} V' + (f' (Ilim={current_limit} A)' if current_limit is not None else ''))
    try:
        psu.set_ch1_voltage(volts, current_limit)
    except AttributeError:
        # Simulator path
        if current_limit is not None:
            psu.write(f'APPL CH1,{volts},{current_limit}')
        else:
            psu.write(f'APPL CH1,{volts}')
    print('Done.')


In [5]:
# EXAMPLES — run the ones you need
set_ch1_voltage(0.0)


Setting CH1 voltage to 0.0 V
Done.


In [6]:
set_ch1_voltage(1.0)


Setting CH1 voltage to 1.0 V
Done.


In [7]:
set_ch1_voltage(2.5, current_limit=0.5)  # example with current limit


Setting CH1 voltage to 2.5 V (Ilim=0.5 A)
Done.


In [8]:
# Cleanup (optional)
try:
    psu.close()
    print('Closed.')
except Exception as e:
    print('Close error:', e)


Closed.
